### Warm-up: How to grab sea surface temperature near Houston.


In [ ]:
import numpy as np

# --- The raw-array way ---
# Pretend we loaded a 720x1440 SST grid.
# Lat runs from -89.875 to 89.875 in 0.25° steps.
# Lon runs from 0.125 to 359.875 in 0.25° steps.

# np.arange() is basically MATLAB linspace(), returns a 1D array of evenly spaced numbers
lats = np.arange(-89.875, 90, 0.25)
lons = np.arange(0.125, 360, 0.25)


# I want lat ≈ 29°N, lon ≈ 265°E (Gulf of Mexico, near Houston)
lat_idx = np.argmin(np.abs(lats - 29.0))
lon_idx = np.argmin(np.abs(lons - 265.0))

print(f'Raw array: lat index={lat_idx}, lon index={lon_idx}')
print(f'That corresponds to lat={lats[lat_idx]:.3f}, lon={lons[lon_idx]:.3f}')
print()



# Lesson 4: Gridded Data with xarray

The warm-up shows the core problem with raw arrays: to select data at a specific location,
you have to find the right integer index yourself. With a 2D array it's annoying.
With a 4D array - time × depth × latitude × longitude - it becomes error-prone fast,
because you also have to remember *which axis is which*.

**xarray** solves this by attaching coordinate labels to every dimension. Once your data
is in an xarray DataArray, you select by value - `lat=29.0` - not by index.

---

### Today's goals

By the end of this session you should be able to:

- Explain the difference between a **DataArray** and a **Dataset**
- Identify the three parts of a DataArray: **dims**, **coords**, and **values**
- Open a NetCDF file with `xr.open_dataset()`
- Select data by position with `.isel()` and by coordinate label with `.sel()`
- Compute reductions over named dimensions: `.mean(dim='lat')`
- Use `.where()` to mask values, and `.plot()` for a quick look

## 1. The xarray data model

xarray has two main objects:

| Object | What it is |
|--------|-----------|
| **DataArray** | One named variable with labeled dimensions |
| **Dataset** | A collection of DataArrays sharing the same coordinates |

A **DataArray** has three things:

```
DataArray
├── values     - the actual numbers (a numpy array under the hood)
├── dims       - the names of the axes: ('time', 'lat', 'lon')
└── coords     - the coordinate labels along each axis:
                  time  → ['2022-03-04']
                  lat   → [-89.875, -89.625, ..., 89.875]
                  lon   → [0.125, 0.375, ..., 359.875]
```

> **Key distinction - `dims` vs `coords`:**
> `dims` are just the *names* of the axes (`'lat'`, `'lon'`, `'time'`).
> `coords` are the *actual values* along each axis (the lat/lon/time arrays).
> A dimension always has a name. It doesn't always have coordinate values -
> but for scientific data it almost always does.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Build a small DataArray from scratch to see the anatomy
# (same idea as building a DataFrame from a dict in Lesson 2)

# Fake 3x4 temperature grid
temps = np.array([[15.1, 16.2, 17.3, 18.0],
                  [20.5, 21.8, 22.1, 23.4],
                  [26.0, 27.2, 28.5, 29.1]])

da = xr.DataArray(
    data   = temps,
    dims   = ['lat', 'lon'],           # axis names
    coords = {'lat': [20, 25, 30],     # coordinate values
               'lon': [260, 265, 270, 275]},
    name   = 'sst',
    attrs  = {'units': 'Celsius', 'long_name': 'Sea Surface Temperature'}
)

da

### Activity 1a: Discussion with neighbors
What happens if we plot the temps array as it is? What do you notice about the plot?
Take 1-2 minutes to discuss what might be odd about this image?
Run the below cell to generate the plot.

In [ ]:
plt.imshow(temps)

### Activity 1b: Discussion with neighbors
What is different is we plot it as a full DataArray?
Take 1-2 minutes to discuss what is different about this image compared to the first one!
Run the below cell to generate the plot.

In [ ]:
# Now look if we plot the DataArray instead of just the temps np array

da.plot()
plt.show()

### Preview: Usage xarray to extract and plot specific parts of a DataArray

In [ ]:
da.sel(lat=25).plot(marker='o')
plt.title('Temperature Along Latitude 25°')
plt.show()

### What does an xarray DataArray actually look like?

In [ ]:
# The three parts xarray used to make that plot
print('dims:  ', da.dims)           # axis names  → became the axis labels
print('coords:', list(da.coords))   # coord names → became the tick values
print('attrs: ', da.attrs)          # metadata    → became the colorbar label
print()
print('The raw numbers (numpy array underneath):')
print(da.values)

### Activity: Build your own DataArray! 2-3 mins

In [ ]:
# ✏️ Try it:
# Build a DataArray for a 3-day time series of a single location's temperature.
# dims = ['time'], coords = {'time': ['2022-01-01', '2022-01-02', '2022-01-03']}
# Give it a 'units' attribute, then plot it with marker='o'.


## 2. Opening a real NetCDF file

In practice you never build DataArrays from scratch - you open files.

The key advantage of xarray: it loads **everything together**. You open the file once and
get all variables and their coordinates in one object. The coordinates stay attached to the
data throughout - no risk of accidentally pairing the wrong variable with the wrong coordinate
array.

We'll use a daily sea surface temperature product from NOAA/AVHRR (OISST v2).
One day of global SST at 0.25° resolution.

In [ ]:
# Open the NetCDF - xarray reads the whole file structure, not just one variable
ds = xr.open_dataset('data/oisst_sst_20220304.nc')

# The repr is interactive in Jupyter - click the arrows to expand
ds

In [ ]:
# Inspect the Dataset structure
print('Dimensions:', dict(ds.sizes))
print('Variables: ', list(ds.data_vars))
print('Coords:    ', list(ds.coords))

# Each variable has its own metadata
print()
for var in ds.data_vars:
    print(f'  {var:6s}: {ds[var].attrs.get("long_name","")} [{ds[var].attrs.get("units","")}]')

### Pull out the SST variable as a DataArray

In [ ]:
# Pull out the SST variable as a DataArray
sst = ds['sst']
print(sst)

### What's up with the 4th dimension? Turns out we don't need it!

It's relatively common for datasets to include a 4th dimension that is more of an identifier than an actual piece of data. Or, sometimes the data is so simple it can be better communicated with just a comment or name. We can remove those unnecesarry dims with squeeze()

In [ ]:
# The SST array has 4 dimensions: (time=1, zlev=1, lat=720, lon=1440)
# time=1 because this file is a single day.
# zlev=1 because this is surface only (z-level = 0m depth).
# Both are size-1 and add structure but no data.
# .squeeze() drops all size-1 dimensions, giving us a clean 2D lat×lon array.

sst_2d = sst.squeeze()
print('Before squeeze:', sst.dims, sst.shape)
print('After squeeze: ', sst_2d.dims, sst_2d.shape)

In [ ]:
# xarray's .plot() auto-titles axes using coordinate names and attrs
# No need to call plt.xlabel() manually
h = sst_2d.plot(cmap='RdYlBu_r', figsize=(12, 5))
plt.title('Global Sea Surface Temperature - March 4, 2022') # set the plot title
h.set_clim(-2, 35) # set the colorbar limits manually

# plt.show() is normally used to make sure python actually renders the image
# jupyter notebook usually does it by default but we include it anyway just in case
plt.show()  



### Activity!! Investigate your dataset. 2-3 mins

In [ ]:
# ✏️ Try it:
# 1. What are the latitude and longitude ranges in this dataset?
# hint: min() and max() will be used, but how do we use functions that affect data objects?

# 2. What is the spatial resolution in degrees?
# hint: look at the difference between consecutive lat values

# 3. Plot the 'ice' variable the same way. What does it show?

# Make sure you are looking at the dataset (ds) not the DataArray (sst_2d)

## 3. Selecting data: `.isel()` and `.sel()`

xarray gives you two selection methods: `.isel()` and `.sel()`. Think of `.isel()` as xarray's version of pandas `.iloc[]`, and `.sel()` as xarray's version of `.loc[]`. The difference is that xarray labels are often physical coordinates (e.g., latitude, longitude, or timestamps) rather than row names, making `.sel()` especially useful for scientific data.

| Method | Selects by | Example |
|--------|-----------|---------|
| `.isel()` | **Integer position** (0, 1, 2...) | `da.isel(lat=100)` |
| `.sel()` | **Coordinate value** | `da.sel(lat=25.0)` |


### Activity! Discussion: Why is this plot blank?

In [ ]:
# .isel(): select by integer index (like plain Python/numpy array indexing)
first_row = sst_2d.isel(lat=0)
print('lat value of row 0:', float(first_row.lat), '°N')

first_row.plot(marker='o', markersize=2, linewidth=0.8)
plt.title(f'SST at lat = {float(first_row.lat):.3f}°N (row 0)')
plt.show()

### Using `.sel()` to select by coordinate value

In [ ]:
# .sel(): select by coordinate VALUE
# method='nearest' finds the closest grid point if the exact value isn't in the grid
equator = sst_2d.sel(lat=0.0, method='nearest')
print('lat value matched:', float(equator.lat), '°N')

equator.plot(marker='o', markersize=2, linewidth=0.8, figsize=(12, 3))
plt.title('Equatorial SST - March 4, 2022')
plt.show()

### Using `.sel()` to select a range of values

In [ ]:
# .sel() with slice() to select a range
# Gulf of Mexico region: lat 18-31°N, lon 262-278°E (0-360 convention)

gulf = sst_2d.sel(
    lat=slice(18, 31),
    lon=slice(262, 278)
)

print('Gulf of Mexico subset:')
print('  Shape:', gulf.shape)
print('  Lat range:', float(gulf.lat.min()), 'to', float(gulf.lat.max()))
print('  Lon range:', float(gulf.lon.min()), 'to', float(gulf.lon.max()))

gulf.plot(cmap='RdYlBu_r', figsize=(8, 5))
plt.title('Gulf of Mexico SST - March 4, 2022')
plt.show()

### Activity!! Debugging. 1 min

In [ ]:
# 🐛 Debugging: What's wrong here? Run it, read the error, then fix it.
# Hint: look at what sel() expects vs what we're giving it.

#broken = sst_2d.sel(lat=29.0, lon=265.0)    # KeyError!

# Fix it below (hint: the exact values 29.0 and 265.0 may not be in the grid):


### Activity!! Subsetting and plotting from a DataArray. 3-ish mins

In [ ]:
# ✏️ Try it:
# 1. Select SST for a region of your choice - a different ocean basin or a coastal area.
#    Plot it. (Reminder: lon is in 0–360 in this file, not -180 to 180)
# 2. What is the SST at the single point closest to Houston, TX?
#    Houston is approximately 29.8°N, 264.6°E (= -95.4° in -180/180 notation)



## 4. Computations over named dimensions

Because dimensions have *names* in xarray, you compute stats and such by name -
no need to remember that axis 0 is lat or axis 1 is lon.

| Operation | xarray |
|-----------|--------|
| Mean over all lon | `da.mean(dim='lon')` |
| Mean over lat + lon | `da.mean(dim=['lat','lon'])` |
| Mean over time | `da.mean(dim='time')` |
| Mask values | `da.where(da > 0)` |

In [ ]:
# Mean SST at each latitude (average over all longitudes)
# This gives us a zonal mean: one value per latitude band
zonal_mean = sst_2d.mean(dim='lon')
print('Input shape: ', sst_2d.shape)    # (lat, lon)
print('Output shape:', zonal_mean.shape) # (lat,) - lon dimension collapsed

zonal_mean.plot()
plt.title('Zonal mean SST (averaged over all longitudes)')
plt.ylabel('SST (°C)')
plt.show()

### Compute over all dims (global mean)

In [ ]:
# Global mean (collapse both lat and lon)
global_mean = sst_2d.mean(dim=['lat', 'lon'])
print(f'Global mean SST: {float(global_mean):.2f} °C')

# NaN values over land are ignored automatically (skipna=True is the default)
print(f'Number of NaN values (land): {int(sst_2d.isnull().sum())}')

### Boolean filtering with DataArrays

In [ ]:
# .where(): keep values where the condition is True; everything else becomes NaN
warm_ocean = sst_2d.where(sst_2d > 20)

h = warm_ocean.plot(cmap='hot_r', figsize=(12, 5))
h.set_clim(20, 35)
plt.title('Ocean areas above 20°C - March 4, 2022')
plt.show()

### Activity!! Filtering, testing stats, and plotting the DataArray

In [ ]:
# ✏️ Try it:
# 1. What is the mean SST in the Gulf of Mexico region we defined earlier?
#    (use gulf.mean(dim=['lat','lon']))
# 2. Compute a meridional mean (average over latitudes, keep longitudes).
#    What does that tell you? What shape is the result?
# 3. Using .where(), create a DataArray showing only SST values below 5°C.
#    Plot it - where in the world is that?



## 5. Working with a Dataset (multiple variables)

A **Dataset** is a collection of DataArrays that share coordinates.
The OISST file has four variables: `sst`, `anom`, `err`, `ice`.
They all live on the same (time, zlev, lat, lon) grid.

You can do computations across variables directly - useful when you want to
combine or compare them. Because the coordinates are shared and always attached,
xarray automatically aligns variables when you do math between them.

In [ ]:
# Squeeze the whole Dataset at once (drops the size-1 time and zlev dims)
ds_2d = ds.squeeze()
print('Dataset variables after squeeze:')
for var in ds_2d.data_vars:
    print(f'  {var}: {ds_2d[var].dims} {ds_2d[var].shape}')

In [ ]:
# The anomaly variable: SST departure from the long-term climatological mean
# Positive = warmer than average, negative = cooler than average

gulf_anom = ds_2d['anom'].sel(lat=slice(18, 31), lon=slice(262, 278))

h = gulf_anom.plot(cmap='RdYlBu_r', figsize=(8, 5))
h.set_clim(-3, 3) # set the colorbar limits manually

plt.title('Gulf of Mexico SST Anomaly - March 4, 2022\n(red = warmer than climatology)')
plt.show()

In [ ]:
# You can do math across variables - they automatically align on shared coordinates
# Reconstruct the climatological mean: sst - anom = climatology
climatology = ds_2d['sst'] - ds_2d['anom']
gulf_clim = climatology.sel(lat=slice(18, 31), lon=slice(262, 278))

print('Gulf mean observed SST:   ', float(ds_2d['sst'].sel(lat=slice(18,31), lon=slice(262,278)).mean()), '°C')
print('Gulf mean climatology:    ', float(gulf_clim.mean()), '°C')
print('Gulf mean anomaly:        ', float(gulf_anom.mean()), '°C')

## 6. Real NASA data: MODIS snow cover (HDF)

So far we've worked with NetCDF. But MODIS products - and many older NASA datasets -
come in HDF4 files. xarray opens them with the same `open_dataset()` call;
you just specify `engine='netcdf4'`.

We have a tile from **MYD10A1F** - MODIS/Aqua **Cloud-Gap-Filled Snow Cover**.
Cloud-gap-filling means the algorithm looks back at recent clear days
to estimate snow cover wherever clouds would otherwise block the view.
The result is a complete spatial picture every day, cloud or not.

This tile is **h11v04** which is mainly Iowa/Illinois/Minnesota/Wisconsin but also covers bits of Ohio/Indiana/Ontario for **February 15, 2026**.

> **SARP connection:** Snow cover drives surface albedo, which affects the surface energy budget
> and boundary layer development. Any project looking at surface–atmosphere exchange
> (aerosol loading, trace gas fluxes, land–air coupling) may need to account for whether
> the surface is snow-covered.

> **Discussion:** Before you look at the data - where would you expect to see the most snow
> in this tile in mid-February? Any areas you'd expect to be patchy or snow-free?

In [ ]:
# Open MODIS HDF4 - same API as NetCDF, just engine='netcdf4'
ds_snow = xr.open_dataset(
    'data/MYD10A1F.A2026046.h11v04.061.2026048035003.hdf',
    engine='netcdf4'
)

# Pull out the cloud-gap-filled snow cover variable
snow = ds_snow['CGF_NDSI_Snow_Cover']
print(snow)
print()
print('Value key:')
print(snow.attrs['Key'])

Notice the dimension names: `YDim:MOD_Grid_Snow_500m` and `XDim:MOD_Grid_Snow_500m`.
That's xarray preserving the HDF metadata, but it's awkward to type.

Also notice: there are **no coordinate values** for those dimensions. MODIS tiles are in
**sinusoidal projection**, not geographic lat/lon. The pixel positions describe a projected grid,
not degrees. For a quick look at the data pattern this is fine; reprojecting to lat/lon is a
separate step.

The other thing to handle: values **above 100 are fill codes**, not snow percentages.
For example, 200 = missing data, 239 = ocean, 250 = cloud. We need to mask those
before computing anything meaningful.

In [ ]:
# Rename the verbose MODIS dimension names to something short
snow = snow.rename({
    'YDim:MOD_Grid_Snow_500m': 'y',
    'XDim:MOD_Grid_Snow_500m': 'x'
})

In [ ]:
# Values 0–100 = NDSI snow cover percent (0 = bare ground, 100 = full snow)
# Values > 100 = fill codes (cloud, water, night, etc.) - mask them
snow_clean = snow.where(snow <= 100)

snow_clean.plot(cmap='Blues_r', figsize=(8, 7))
plt.title('MODIS Aqua Cloud-Gap-Filled Snow Cover\nTile h11v04 (Upper Midwest US) - Feb 15, 2026')
plt.xlabel('x pixel')
plt.ylabel('y pixel')
plt.show()

# Quick summary
n_valid = int(snow_clean.count())
n_total = snow.size
print(f'Valid (snow or bare ground) pixels: {n_valid:,} of {n_total:,} ({100*n_valid/n_total:.0f}%)')

## 7. Bridging xarray and geopandas: adding coordinates to MODIS

Right now our MODIS DataArray has pixel indices for dimensions, which have no geographic meaning.
To overlay lakes, states, or other map data, the snow grid and the vector data need to be
in the same coordinate reference system.

This tile uses the **MODIS sinusoidal projection**. The metadata tells us the upper-left
corner and pixel size in projected meters. We can turn those projected
x/y positions into longitude/latitude with `pyproj`.

One important wrinkle: longitude is not a single 1D coordinate for this sinusoidal tile.
Each row has a slightly different lon spacing, so we will attach **2D lon/lat coordinates**
to the xarray DataArray before plotting.

In [ ]:
from pyproj import CRS, Transformer

# Projection/grid parameters from the HDF metadata
R = 6371007.181          # MODIS sinusoidal sphere radius (m)
x_ul = -7783653.637667   # upper-left X corner (m)
y_ul =  5559752.598333   # upper-left Y corner (m)
ps = 463.312716527778    # pixel size (m)

# Pixel-center coordinates in the native MODIS sinusoidal grid
n_y = snow_clean.sizes['y']
n_x = snow_clean.sizes['x']
x_centers = x_ul + (np.arange(n_x) + 0.5) * ps
y_centers = y_ul - (np.arange(n_y) + 0.5) * ps
x2d, y2d = np.meshgrid(x_centers, y_centers)

# Convert MODIS sinusoidal meters to lon/lat degrees
modis_crs = CRS.from_proj4(f'+proj=sinu +R={R} +nadgrids=@null +wktext')
lonlat_crs = CRS.from_epsg(4326)
transformer = Transformer.from_crs(modis_crs, lonlat_crs, always_xy=True)
lon2d, lat2d = transformer.transform(x2d, y2d)

# Attach the 2D coordinates to the same xarray DataArray
snow_geo = snow_clean.assign_coords(
    lon=(('y', 'x'), lon2d),
    lat=(('y', 'x'), lat2d)
)

# Also attach the native projected x/y coordinates. These are useful for transformations,
# but the final figure below will use pixel coordinates to preserve the original tile shape.
snow_map = snow_clean.assign_coords(
    x=('x', x_centers),
    y=('y', y_centers)
)

print(f'Lat range: {np.nanmin(lat2d):.2f}° to {np.nanmax(lat2d):.2f}°N')
print(f'Lon range: {np.nanmin(lon2d):.2f}° to {np.nanmax(lon2d):.2f}°E')
print('snow_geo dims:', snow_geo.dims)
print('snow_geo coords:', list(snow_geo.coords))
print('snow_map coords:', list(snow_map.coords))

In [ ]:
# Make a MATLAB-style parula colormap
from matplotlib.colors import LinearSegmentedColormap

# Sample subset of official MATLAB Parula RGB values
parula_data = [
    (0.2081, 0.1663, 0.5292),
    (0.0591, 0.3598, 0.8683),
    (0.0231, 0.6418, 0.7913),
    (0.1801, 0.7177, 0.6424),
    (0.5300, 0.7491, 0.4661),
    (0.9139, 0.7258, 0.3063),
    (0.9763, 0.9831, 0.0538)
]

parula_cmap = LinearSegmentedColormap.from_list('Parula', parula_data)

In [ ]:
import geopandas as gpd
import cartopy.io.shapereader as shpreader

# Load Great Lakes from Natural Earth (cached locally by cartopy after first download)
lakes_shp = shpreader.natural_earth(resolution='10m', category='physical', name='lakes')
lakes = gpd.read_file(lakes_shp)
great_lakes = lakes[lakes['name'].isin([
    'Lake Superior', 'Lake Michigan', 'Lake Huron', 'Lake Erie', 'Lake Ontario'
])]

# Plot: xarray handles the snow data, geopandas handles the lake outlines
fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)

mesh = snow_geo.plot.pcolormesh(
    ax=ax,
    x='lon',
    y='lat',
    cmap=parula_cmap,
    vmin=0,
    vmax=85,
    add_colorbar=False,
    rasterized=True
)
great_lakes.plot(ax=ax, facecolor='lightcyan', edgecolor='steelblue', linewidth=1.5,
                 zorder=3, label='Great Lakes')

ax.set_title('MODIS Cloud-Gap-Filled Snow Cover with Great Lakes\nTile h11v04 - Feb 15, 2026')
ax.set_xlabel('Longitude (°)')
ax.set_ylabel('Latitude (°)')
ax.set_xlim(np.nanmin(lon2d), np.nanmax(lon2d))
ax.set_ylim(np.nanmin(lat2d), np.nanmax(lat2d))
ax.set_aspect('equal')

cbar = fig.colorbar(mesh, ax=ax, orientation='horizontal', pad=0.08,
                    fraction=0.055, aspect=40)
cbar.set_label('Snow cover (%)')

# Optional: try Natural Earth state/province lines too
# states_shp = shpreader.natural_earth(resolution='50m', category='cultural', name='admin_1_states_provinces_lines')
# gpd.read_file(states_shp).plot(ax=ax, color='0.25', linewidth=0.4, zorder=4)
plt.show()

## Wrapping up

The three things that matter most from today:

1. **xarray keeps data and coordinates together.** A DataArray has `values` (the numbers),
   `dims` (axis names), and `coords` (the actual coordinate values). You never have to
   separately track "which array is my lat vector" - it's always attached.

2. **Select by label, not by index.** `.sel(lat=29.0, method='nearest')` instead of
   finding the index yourself. This is the single biggest quality-of-life improvement
   over working with raw arrays.

3. **Computations use dimension names.** `.mean(dim='lat')` instead of counting axes.
   No silent wrong answers from mixing up axis 0 and axis 1.

Everything else - `.isel()`, `.where()`, `.squeeze()`, `rename()`, opening HDF vs NetCDF -
follows from those three ideas. You'll pick it up as you go.

---

### Exit ticket

1. Open the OISST dataset and squeeze out the size-1 dimensions.
2. Select SST for the North Atlantic: lat 30–65°N, lon 280–360°E.
3. Compute the mean SST for that region.
4. Using `.where()`, show only grid points above the regional mean. Plot it.
5. In one sentence: what is the difference between `.isel(lat=0)` and `.sel(lat=0.0, method='nearest')`?

In [ ]:
# Exit ticket - write your answers here:

# 1 & 2.

# 3.

# 4.

# 5. (answer as a comment)

In [ ]:
# Exit ticket - answers:

# 1 & 2. Open, squeeze, subset North Atlantic
ds_et = xr.open_dataset('data/oisst_sst_20220304.nc').squeeze()
natl = ds_et['sst'].sel(lat=slice(30, 65), lon=slice(280, 360))

# 3. Regional mean
natl_mean = float(natl.mean())
print(f'North Atlantic mean SST: {natl_mean:.2f} °C')

# 4. Plot above-mean pixels
natl.where(natl > natl_mean).plot(cmap='RdYlBu_r', figsize=(10, 5))
plt.title(f'N. Atlantic SST above regional mean ({natl_mean:.1f} °C)')
plt.show()

# 5. .isel(lat=0) selects the row at integer position 0 - the first row in the array,
#    regardless of what coordinate value that corresponds to (-89.875°N here).
#    .sel(lat=0.0, method='nearest') finds the row whose lat coordinate is closest to 0.0°N.